## 1. Data Cleaning Pipeline — Mouad

I created cleaning functions to handle some of the main problems in the transaction data. These include duplicate records, inconsistent store IDs, mixed timestamp formats, prices stored as text, and negative quantities.

I wanted to clean the data without removing more information than necessary. For example, I removed exact duplicates but kept transactions with missing timestamps or negative quantities and flagged them for further review. I also kept the original columns so we can compare the cleaned values with the raw data later.

In [1]:
import pandas as pd
from pathlib import Path

def find_data(filename):
    for folder in ["data", "../data", "/mnt/data"]:
        path = Path(folder) / filename
        if path.exists():
            return path
    raise FileNotFoundError(filename)

raw = pd.read_csv(find_data("walmart_transactions.csv"))
stores_raw = pd.read_csv(find_data("store_locations.csv"))

def clean_store_id(series):
    return (
        series.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(
            r"^FL[-_\s]*(\d+)$",
            r"FL-\1",
            regex=True
        )
    )

def clean_dates(series):
    # Explicit formats prevent month/day confusion.
    formats = [
        "%Y-%m-%dT%H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%m/%d/%Y %H:%M",
        "%m-%d-%Y %H:%M:%S",
        "%d-%b-%Y %H:%M"
    ]

    result = pd.Series(
        pd.NaT, index=series.index, dtype="datetime64[ns]"
    )

    for fmt in formats:
        parsed = pd.to_datetime(
            series, format=fmt, errors="coerce"
        )
        result = result.fillna(parsed)

    return result

def clean_transactions(df):
    cleaned = df.copy()
    log = []

    # 1. Remove exact duplicates
    before = len(cleaned)
    cleaned = cleaned.drop_duplicates().copy()

    log.append({
        "step": "Remove duplicates",
        "before": before,
        "after": len(cleaned),
        "affected": before - len(cleaned)
    })

    # 2. Standardize store IDs
    cleaned["store_id_clean"] = clean_store_id(
        cleaned["store_id"]
    )

    changed = (
            cleaned["store_id"].astype("string")
            != cleaned["store_id_clean"]
    ).sum()

    log.append({
        "step": "Standardize store IDs",
        "before": len(cleaned),
        "after": len(cleaned),
        "affected": int(changed)
    })

    # 3. Parse mixed timestamps
    cleaned["timestamp_clean"] = clean_dates(
        cleaned["timestamp"]
    )

    cleaned["date"] = cleaned["timestamp_clean"].dt.date

    log.append({
        "step": "Flag missing timestamps",
        "before": len(cleaned),
        "after": len(cleaned),
        "affected": int(
            cleaned["timestamp_clean"].isna().sum()
        )
    })

    # 4. Convert unit prices into numbers
    price_text = cleaned["unit_price"].astype("string")

    formatted_prices = price_text.str.contains(
        r"\$|^\s|\s$", regex=True, na=False
    ).sum()

    cleaned["unit_price_clean"] = pd.to_numeric(
        price_text
        .str.strip()
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False),
        errors="coerce"
    )

    log.append({
        "step": "Clean unit prices",
        "before": len(cleaned),
        "after": len(cleaned),
        "affected": int(formatted_prices)
    })

    # 5. Convert quantities and flag possible returns
    cleaned["quantity_clean"] = pd.to_numeric(
        cleaned["quantity"], errors="coerce"
    )

    cleaned["possible_return"] = (
            cleaned["quantity_clean"] < 0
    )

    log.append({
        "step": "Flag negative quantities",
        "before": len(cleaned),
        "after": len(cleaned),
        "affected": int(cleaned["possible_return"].sum())
    })

    # 6. Prepare standardized SKU and category values
    cleaned["sku_clean"] = (
        cleaned["sku"].astype("string")
        .str.strip().str.upper()
    )

    cleaned["category_clean"] = (
        cleaned["category"].astype("string")
        .str.strip().str.title()
    )

    return cleaned, pd.DataFrame(log)

In [2]:
# Clean transactions
clean, decision_log = clean_transactions(raw)

# Standardize the store reference file
stores_clean = stores_raw.copy()
stores_clean["store_id_clean"] = clean_store_id(
    stores_clean["store_id"]
)

# Check duplicated store IDs do not conflict
conflicts = stores_clean.groupby("store_id_clean")[
    ["city", "latitude", "longitude"]
].nunique(dropna=True)

assert not conflicts.gt(1).any().any()

stores_clean = stores_clean.drop_duplicates(
    subset="store_id_clean"
)

print("Original transactions:", len(raw))
print("Cleaned transactions:", len(clean))
print("Removed:", len(raw) - len(clean))
print("Unique stores:", clean["store_id_clean"].nunique())
print()

display(decision_log)

# Basic verification
assert len(clean) == 25532
assert clean["unit_price_clean"].notna().all()
assert clean["quantity_clean"].notna().all()
assert clean["store_id_clean"].isin(
    stores_clean["store_id_clean"]
).all()

print("Initial cleaning checks passed!")

Original transactions: 25562
Cleaned transactions: 25532
Removed: 30
Unique stores: 12



,step,before,after,affected
0,Remove duplicates,25562,25532,30
1,Standardize store IDs,25532,25532,4345
2,Flag missing timestamps,25532,25532,884
3,Clean unit prices,25532,25532,4329
4,Flag negative quantities,25532,25532,130


Initial cleaning checks passed!


## 2. Cleaning Decision Log

We started with 25,562 transaction records. After removing 30 exact duplicates, we had 25,532 records remaining. For the other cleaning steps, we decided to keep the rows and either standardize or flag the problematic values rather than automatically removing them.

| Problem | Decision | Rows affected | Reason |
|---|---|---:|---|
| Duplicate transactions | Remove exact duplicates | 30 | These could inflate sales if counted more than once. |
| Inconsistent store IDs | Standardize capitalization, spaces, and separators | 4,345 | Different formats for the same store could cause incorrect joins. |
| Missing timestamps | Keep and flag the missing dates | 884 | We shouldn't guess when purchases happened. These rows can be excluded from date-based analysis. |
| Prices stored as text | Remove formatting and convert to numeric | 4,329 | We need numeric prices for accurate revenue calculations. |
| Negative quantities | Keep and flag them as possible returns | 130 | These might represent returns instead of errors, so we should investigate before removing them. |
| Inconsistent product information | Create standardized SKU and category fields, and start a mapping table | Mapping in progress | Different codes and names might refer to the same product. |

### Verification

After cleaning, the dataset contained 25,532 transactions. All unit prices and quantities converted successfully, and the cleaned store IDs matched the standardized store reference data.

One thing we still need to investigate is whether negative quantities represent actual returns. We also need to finish checking the SKU mapping, especially when different SKUs have different package sizes.

## 3. SKU Consolidation — Pop-Tarts First

I created a mapping table that assigns all four codes to the Strawberry Pop-Tarts product family while keeping their original SKUs.

One issue I noticed is that the package sizes are not all the same. Three SKUs describe four-packs, while one describes a six-pack. Because of this, I don't want to treat all four SKUs as identical units without adjusting for package size. For now, the mapping groups them under one family but preserves their differences for later analysis.

In [3]:
# Load product catalog
products = pd.read_csv(find_data("product_catalog.csv"))

# Standardize catalog SKU values
products["sku_clean"] = (
    products["sku"]
    .astype("string")
    .str.strip()
    .str.upper()
)

# Start SKU mapping with Strawberry Pop-Tarts
pop_tarts_mapping = pd.DataFrame({
    "sku_clean": [
        "PT-12",
        "SKU#459812",
        "POPTART-STRAW",
        "PT_STRAWBERRY"
    ],
    "product_family": [
                          "Strawberry Pop-Tarts"
                      ] * 4
})

# Add the original package information
pop_tarts_mapping = pop_tarts_mapping.merge(
    products[[
        "sku_clean",
        "product_name",
        "package_size"
    ]],
    on="sku_clean",
    how="left",
    validate="one_to_one"
)

assert len(pop_tarts_mapping) == 4

display(pop_tarts_mapping)

,sku_clean,product_family,product_name,package_size
0,PT-12,Strawberry Pop-Tarts,pop-tarts strawberry,4-pack
1,SKU#459812,Strawberry Pop-Tarts,Strawberry Pop-Tart,4pk
2,POPTART-STRAW,Strawberry Pop-Tarts,Pop-Tarts Strawberry,6 pack
3,PT_STRAWBERRY,Strawberry Pop-Tarts,pop-tarts strawberry,4 pack


In [4]:
# Get transactions for the four Pop-Tarts SKUs
pop_sales = clean[
    clean["sku_clean"].isin(
        pop_tarts_mapping["sku_clean"]
    )
    & (clean["quantity_clean"] > 0)
    ].copy()

# Sum positive quantities separately by SKU
sku_totals = (
    pop_sales.groupby("sku_clean")["quantity_clean"]
    .sum()
    .sort_values(ascending=False)
)

print("Positive quantities by SKU:")
display(sku_totals.to_frame("quantity_sold"))

print("Combined Pop-Tarts quantity:", sku_totals.sum())
print("Number of SKUs:", sku_totals.size)

Positive quantities by SKU:


,quantity_sold
sku_clean,
PT-12,1381
POPTART-STRAW,1300
PT_STRAWBERRY,1262
SKU#459812,1255


Combined Pop-Tarts quantity: 5198
Number of SKUs: 4


In [5]:
# Confirm all four Pop-Tarts SKUs are represented
assert pop_tarts_mapping["sku_clean"].nunique() == 4
assert pop_tarts_mapping["product_family"].nunique() == 1

# Confirm the positive sales total
assert sku_totals.sum() == 5198

print("SKU mapping checks passed!")

SKU mapping checks passed!


### Initial SKU Mapping Findings

After grouping the four Strawberry Pop-Tarts SKUs, I found 5,198 reported positive-quantity units sold across the four codes. This shows why consolidating related SKUs matters: looking at just one code would miss a large portion of the sales.

However, some SKUs have different package sizes, so combining their quantities directly could be misleading. I kept the original SKU and package information in the mapping table so we can review those differences before using the combined numbers for our final demand analysis. For now, I have completed the initial Pop-Tarts mapping.